# NB02c — Downstream: ESMFold → ProteinMPNN → Protenix → Filter

Takes a `forge_sequences.csv` (from NB02c Forge generation step) and runs the full
structural validation pipeline.

## Pipeline
```
forge_sequences.csv  (upload)
    ↓  fix names if needed
ESMFold pass 1  →  binder backbone PDBs + pLDDT filter (≥70)
    ↓
ProteinMPNN (monomer)  →  4 redesigned sequences per backbone
    ↓
free GPU (del esmfold)
    ↓
Protenix (binder + K18 complex, no MSA)  →  ipTM + ranking_score
    ↓
Epitope contact analysis on CIF output
    ↓
Filter: ranking_score ≥ 0.5 AND epitope_contacts ≥ 1
    ↓
forge_binder_filter_passing.csv + CIFs  ← feed into NB03
```
**Requires**: GPU runtime (T4, ≥16 GB)
**Runtime**: ~1.5–2 h (no Forge generation)

## 0. GPU check

In [ ]:
import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else 'No GPU — switch to GPU runtime')

In [ ]:
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_BASE = '/content/drive/MyDrive/CHEM_280'
    _in_drive  = True
    print(f'Drive mounted. Base: {DRIVE_BASE}')
except Exception:
    DRIVE_BASE = '/content'
    _in_drive  = False
    print('Drive not available — saving to /content/')

NB02C_OUT = os.path.join(DRIVE_BASE, 'results', 'nb02c_output')
os.makedirs(NB02C_OUT, exist_ok=True)
print(f'NB02c output dir: {NB02C_OUT}')

## 1. Install

### 1a. ESMFold — sokrypton Colab-patched forks

In [ ]:
%%capture install_esm_out
import os
os.chdir('/content')
!pip install -q omegaconf pytorch_lightning biopython ml_collections einops modelcif
!pip install -q git+https://github.com/NVIDIA/dllogger.git
!pip install -q git+https://github.com/sokrypton/openfold.git
!pip install -q git+https://github.com/sokrypton/esm.git
print('ESMFold ready.')

### 1b. ProteinMPNN

In [ ]:
%%capture install_mpnn_out
!git clone -q https://github.com/dauparas/ProteinMPNN /content/ProteinMPNN
print('ProteinMPNN ready.')

### 1c. Protenix

In [ ]:
%%capture install_protenix_out
import os
!git clone -q https://github.com/bytedance/protenix.git /content/protenix
os.chdir('/content/protenix')
!pip install -q -e .
os.chdir('/content')
print('Protenix ready.')

In [ ]:
import subprocess, os

try:
    import esm
    print('ESMFold    : OK')
except Exception as e:
    print(f'ESMFold    : ERROR — {e}')

mpnn_ok = os.path.exists('/content/ProteinMPNN/protein_mpnn_run.py')
print(f'ProteinMPNN: {"OK" if mpnn_ok else "MISSING — check git clone"}')

r = subprocess.run(['protenix', '--help'], capture_output=True, text=True)
prot_ok = r.returncode == 0 or 'usage' in (r.stdout + r.stderr).lower()
print(f'Protenix   : {"OK" if prot_ok else "check install"}')

## 2. Configuration

In [ ]:
import os

# ── Target sequence ────────────────────────────────────────────────────────────
TARGET_SEQUENCE = (
    'KVQIINKKLDLSNVQSKCGSKDNIKHVPGGGSVQIVYKPVDLSKVTSKCGSLGNIHHKPGGG'
    'QVEVKSEKLDFKDRVQSKIGSLDNITHVPGGGNKKIETHKLTFRENAKAKTDHGAEIVYKSPV'
    'VSGDTSPRHLSNVSSTGSIDMVDSPQLATLADEVSASLAKQGL'
)

# ── Epitope ranges (0-indexed on K18 sequence) ────────────────────────────────
EPITOPE_RANGES = [
    (1,   7,  'PHF6star_VQIINK', 1.0),
    (32,  38, 'PHF6_VQIVYK',     1.0),
    (51,  70, 'jR2R3',           0.5),
]

# ── ESMFold pass 1 ────────────────────────────────────────────────────────────
PLDDT_THRESHOLD      = 70.0

# ── ProteinMPNN ───────────────────────────────────────────────────────────────
MPNN_SEQS_PER_STRUCT = 4
MPNN_SAMPLING_TEMP   = 0.1

# ── Protenix ──────────────────────────────────────────────────────────────────
PROTENIX_MODEL       = 'protenix_mini_default_v0.5.0'
MIN_RANKING_SCORE    = 0.5
MIN_EPITOPE_CONTACTS = 1
CONTACT_DIST_A       = 5.0

# ─────────────────────────────────────────────────────────────────────────────
for d in ['esm1_pdbs', 'mpnn_seqs', 'protenix_runs', 'passing']:
    os.makedirs(f'/content/{d}', exist_ok=True)

print(f'Target length   : {len(TARGET_SEQUENCE)} aa')
print(f'pLDDT threshold : {PLDDT_THRESHOLD}')
print(f'MPNN seqs/struct: {MPNN_SEQS_PER_STRUCT}')
print(f'Protenix model  : {PROTENIX_MODEL}')
print(f'Min ranking     : {MIN_RANKING_SCORE}')

## 3. Load Forge sequences

In [ ]:
import pandas as pd, re, os

# ── Option A: upload from local machine ──────────────────────────────────────
try:
    from google.colab import files as _gfiles
    print('Upload your forge_sequences.csv:')
    uploaded = _gfiles.upload()
    csv_path = list(uploaded.keys())[0]
except Exception:
    # Option B: already on Drive or /content/
    csv_path = os.path.join(DRIVE_BASE, 'results', 'nb02c_output', 'forge_sequences.csv')
    if not os.path.exists(csv_path):
        csv_path = '/content/forge_sequences.csv'
    print(f'Loading from: {csv_path}')

forge_df = pd.read_csv(csv_path)

# ── Fix name column if it has the f-string formatting bug ─────────────────────
# Bug: the training script produced names like "forge_L{}{:04d}" (unevaluated)
_name_ok = forge_df['name'].str.match(r'^forge_L\d+_\d+$', na=False).all()
if not _name_ok:
    print(f'Name column has formatting bug (e.g. "{forge_df["name"].iloc[0]}") — rebuilding...')
    forge_df['name'] = [
        f"forge_L{int(row['length'])}_{i+1:04d}"
        for i, row in forge_df.iterrows()
    ]
    print('Names repaired.')

# ── Validate required columns ─────────────────────────────────────────────────
required = {'name', 'sequence', 'length'}
missing = required - set(forge_df.columns)
if missing:
    raise ValueError(f'CSV missing required columns: {missing}')

# Add method/target_conformer if absent (older CSVs may not have them)
if 'method' not in forge_df.columns:
    forge_df['method'] = 'Forge'
if 'target_conformer' not in forge_df.columns:
    forge_df['target_conformer'] = 'tau_k18'

print(f'\nLoaded {len(forge_df)} sequences')
print(forge_df.groupby('length').size().rename('count').to_frame().to_string())
print(f'\nSample names: {forge_df["name"].head(3).tolist()}')
print(f'Sample sequences (first 20 aa): {forge_df["sequence"].head(3).str[:20].tolist()}')

## 4. ESMFold pass 1 — fold Forge sequences

In [ ]:
import torch, numpy as np, esm

print('Loading ESMFold...')
esmfold = esm.pretrained.esmfold_v1().eval().cuda()
print('ESMFold ready.')

def fold_and_plddt(seq, out_path):
    """Fold sequence, write PDB, return mean pLDDT."""
    with torch.no_grad():
        pdb_str = esmfold.infer_pdb(seq)
    with open(out_path, 'w') as f:
        f.write(pdb_str)
    vals = [float(l[60:66]) for l in pdb_str.splitlines() if l.startswith('ATOM')]
    return np.mean(vals) if vals else 0.0

plddt1, pdbs1 = [], []
print(f'ESMFold pass 1: {len(forge_df)} sequences...')
for i, row in forge_df.iterrows():
    out = f"/content/esm1_pdbs/{row['name']}.pdb"
    plddt1.append(fold_and_plddt(row['sequence'], out))
    pdbs1.append(out)
    if (i + 1) % 50 == 0:
        print(f'  {i+1}/{len(forge_df)}')

forge_df['plddt1'] = plddt1
forge_df['pdb1']   = pdbs1

pass1 = forge_df[forge_df['plddt1'] >= PLDDT_THRESHOLD].copy()
print(f'\nPass 1 pLDDT >= {PLDDT_THRESHOLD}: {len(pass1)}/{len(forge_df)}')

## 5. ProteinMPNN — redesign sequences on ESMFold backbones

In [ ]:
import subprocess, glob as _glob, json as _json, os

def run_mpnn_monomer(pdb_path, out_dir, n_seqs=MPNN_SEQS_PER_STRUCT,
                     temp=MPNN_SAMPLING_TEMP):
    """Run ProteinMPNN on a binder-only PDB — all residues designed."""
    stem    = os.path.splitext(os.path.basename(pdb_path))[0]
    run_dir = os.path.join(out_dir, stem)
    os.makedirs(run_dir, exist_ok=True)

    jsonl = os.path.join(run_dir, 'input.jsonl')
    with open(jsonl, 'w') as f:
        f.write(_json.dumps({'PDB': pdb_path}) + '\n')

    subprocess.run([
        'python', '/content/ProteinMPNN/protein_mpnn_run.py',
        '--jsonl_path',         jsonl,
        '--out_folder',         run_dir,
        '--num_seq_per_target', str(n_seqs),
        '--sampling_temp',      str(temp),
        '--batch_size',         '1',
    ], capture_output=True, text=True)

    seqs = []
    for fa in _glob.glob(os.path.join(run_dir, 'seqs', '*.fa')):
        with open(fa) as fh:
            lines = fh.readlines()
        for idx, line in enumerate(lines):
            if line.startswith('>') and idx + 1 < len(lines):
                seqs.append(lines[idx + 1].strip())
    return seqs

mpnn_records = []
print(f'ProteinMPNN on {len(pass1)} backbones ({MPNN_SEQS_PER_STRUCT} seqs each)...')
for _, row in pass1.iterrows():
    seqs = run_mpnn_monomer(row['pdb1'], '/content/mpnn_seqs')
    for i, seq in enumerate(seqs):
        mpnn_records.append({
            'name':             f"{row['name']}_mpnn_{i+1:02d}",
            'sequence':         seq,
            'forge_parent':     row['name'],
            'method':           'Forge',
            'target_conformer': row['target_conformer'],
        })

mpnn_df = pd.DataFrame(mpnn_records)
print(f'ProteinMPNN sequences ready: {len(mpnn_df)}')

del esmfold
torch.cuda.empty_cache()
print('ESMFold unloaded — GPU freed for Protenix.')

## 6. Protenix — predict binder + K18 complex

In [ ]:
def run_protenix(name, binder_seq, target_seq, out_base, model=PROTENIX_MODEL):
    """
    Predict binder+target complex with Protenix.
    Returns (ranking_score, iptm, cif_path) or (None, None, None) on failure.
    Chain A = target (K18), Chain B = binder.
    """
    run_dir = os.path.join(out_base, name)
    os.makedirs(run_dir, exist_ok=True)

    input_data = [{
        'name': name,
        'sequences': [
            {'proteinChain': {'sequence': target_seq, 'count': 1}},
            {'proteinChain': {'sequence': binder_seq, 'count': 1}},
        ]
    }]
    input_json = os.path.join(run_dir, 'input.json')
    with open(input_json, 'w') as f:
        _json.dump(input_data, f)

    subprocess.run([
        'protenix', 'pred',
        '-i', input_json,
        '-o', run_dir,
        '-n', model,
        '-s', '101',
        '-e', '1',
        '--use_msa', 'false',
        '--use_default_params', 'true',
    ], capture_output=True, text=True, timeout=600)

    conf_files = _glob.glob(os.path.join(run_dir, '**', '*confidence*.json'), recursive=True)
    if not conf_files:
        return None, None, None

    with open(conf_files[0]) as f:
        conf = _json.load(f)
    iptm    = conf.get('iptm', 0.0)
    ptm     = conf.get('ptm',  0.0)
    ranking = 0.8 * iptm + 0.2 * ptm

    cif_files = _glob.glob(os.path.join(run_dir, '**', '*.cif'), recursive=True)
    cif_path  = cif_files[0] if cif_files else None

    return ranking, iptm, cif_path

print('Protenix helper defined.')

In [ ]:
from Bio.PDB import MMCIFParser
import warnings, numpy as np
from Bio.PDB.PDBExceptions import PDBConstructionWarning

def epitope_contacts_from_cif(cif_path, epitope_ranges=EPITOPE_RANGES,
                               cutoff=CONTACT_DIST_A):
    """
    Count binder residues within cutoff Å of any epitope residue.
    Chain A = target (K18), Chain B = binder.
    """
    if cif_path is None or not os.path.exists(cif_path):
        return 0, {}

    with warnings.catch_warnings():
        warnings.simplefilter('ignore', PDBConstructionWarning)
        parser = MMCIFParser(QUIET=True)
        structure = parser.get_structure('complex', cif_path)

    model_s = structure[0]
    chains  = list(model_s.get_chains())
    if len(chains) < 2:
        return 0, {}

    chain_A = chains[0]
    chain_B = chains[1]

    target_residues = list(chain_A.get_residues())
    binder_atoms    = list(chain_B.get_atoms())

    breakdown = {}
    total_set = set()

    for start, end, label, weight in epitope_ranges:
        epi_atoms = []
        for res in target_residues[start:end]:
            epi_atoms.extend(res.get_atoms())

        contacts = set()
        for e_atom in epi_atoms:
            for b_atom in binder_atoms:
                if np.linalg.norm(e_atom.coord - b_atom.coord) < cutoff:
                    contacts.add(b_atom.get_parent().get_id()[1])

        breakdown[label] = len(contacts)
        total_set |= contacts

    return len(total_set), breakdown

print('Epitope contact function defined.')

In [ ]:
protenix_records = []
total = len(mpnn_df)
print(f'Running Protenix on {total} sequences...')

for i, row in mpnn_df.iterrows():
    ranking, iptm, cif_path = run_protenix(
        name       = row['name'],
        binder_seq = row['sequence'],
        target_seq = TARGET_SEQUENCE,
        out_base   = '/content/protenix_runs',
    )

    if ranking is None:
        contacts, breakdown = 0, {}
        ranking = iptm = 0.0
    else:
        contacts, breakdown = epitope_contacts_from_cif(cif_path)

    protenix_records.append({
        **row.to_dict(),
        'ranking_score':     ranking,
        'iptm':              iptm,
        'epitope_contacts':  contacts,
        'epitope_breakdown': str(breakdown),
        'cif_path':          cif_path,
    })

    if (i + 1) % 20 == 0:
        print(f'  {i+1}/{total}  |  last ranking={ranking:.3f}  contacts={contacts}')

results_df = pd.DataFrame(protenix_records)
print(f'\nProtenix done. Median ranking score: {results_df["ranking_score"].median():.3f}')

## 7. Filter

In [ ]:
mask = (
    (results_df['ranking_score']    >= MIN_RANKING_SCORE) &
    (results_df['epitope_contacts'] >= MIN_EPITOPE_CONTACTS)
)
passing = results_df[mask].copy()

print(f'Ranking >= {MIN_RANKING_SCORE} AND contacts >= {MIN_EPITOPE_CONTACTS}:')
print(f'  Passing: {len(passing)} / {len(results_df)}')
if len(passing):
    print(f'  Ranking  : {passing["ranking_score"].min():.3f} - {passing["ranking_score"].max():.3f}')
    print(f'  ipTM     : {passing["iptm"].min():.3f} - {passing["iptm"].max():.3f}')
    print(f'  Contacts : {passing["epitope_contacts"].min()} - {passing["epitope_contacts"].max()}')
else:
    print('  No passing sequences — consider relaxing thresholds.')

## 8. QC plots

In [ ]:
import matplotlib.pyplot as plt, shutil

fig, axes = plt.subplots(2, 3, figsize=(16, 8))

ax = axes[0, 0]
ax.hist(forge_df['plddt1'], bins=20, color='steelblue', edgecolor='white')
ax.axvline(PLDDT_THRESHOLD, color='tomato', ls='--', lw=2, label=f'>={PLDDT_THRESHOLD}')
ax.set_xlabel('pLDDT'); ax.set_title('ESMFold pass 1 (Forge seqs)'); ax.legend(fontsize=8)

ax = axes[0, 1]
ax.hist(results_df['ranking_score'], bins=25, color='darkorange', edgecolor='white')
ax.axvline(MIN_RANKING_SCORE, color='tomato', ls='--', lw=2, label=f'>={MIN_RANKING_SCORE}')
ax.set_xlabel('Ranking score (0.8*ipTM + 0.2*pTM)')
ax.set_title('Protenix ranking scores'); ax.legend(fontsize=8)

ax = axes[0, 2]
ax.hist(results_df['iptm'], bins=25, color='seagreen', edgecolor='white')
ax.set_xlabel('ipTM'); ax.set_title('Protenix ipTM')

ax = axes[1, 0]
ax.hist(results_df['epitope_contacts'], bins=range(0, 30), color='mediumpurple', edgecolor='white')
ax.axvline(MIN_EPITOPE_CONTACTS - 0.5, color='tomato', ls='--', lw=2)
ax.set_xlabel('Epitope contacts (# binder residues)'); ax.set_title('Epitope contacts')

ax = axes[1, 1]
ax.scatter(results_df['epitope_contacts'], results_df['ranking_score'],
           alpha=0.4, s=15, color='slategray')
if len(passing):
    ax.scatter(passing['epitope_contacts'], passing['ranking_score'],
               alpha=0.8, s=25, color='tomato', label='Passing')
ax.axhline(MIN_RANKING_SCORE, color='tomato', ls='--', lw=1)
ax.axvline(MIN_EPITOPE_CONTACTS - 0.5, color='tomato', ls='--', lw=1)
ax.set_xlabel('Epitope contacts'); ax.set_ylabel('Ranking score')
ax.set_title('Ranking vs contacts'); ax.legend(fontsize=8)

ax = axes[1, 2]
if len(passing):
    ax.hist(passing['sequence'].str.len(), bins=15, color='steelblue', edgecolor='white')
else:
    ax.text(0.5, 0.5, 'No passing sequences', ha='center', va='center', transform=ax.transAxes)
ax.set_xlabel('Binder length (aa)'); ax.set_title(f'Passing lengths (n={len(passing)})')

plt.suptitle('Forge -> ProteinMPNN -> Protenix pipeline QC', fontsize=13)
plt.tight_layout()
plt.savefig('/content/forge_pipeline_qc.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: forge_pipeline_qc.png')

## 9. Save — CSV + CIF files

In [ ]:
import shutil

for _, row in passing.iterrows():
    if row['cif_path'] and os.path.exists(row['cif_path']):
        shutil.copy2(row['cif_path'], f"/content/passing/{row['name']}.cif")

out_cols = ['name', 'sequence', 'iptm', 'ranking_score', 'method',
            'target_conformer', 'forge_parent', 'epitope_contacts', 'epitope_breakdown']
out_cols = [c for c in out_cols if c in passing.columns]
passing[out_cols].to_csv('/content/forge_binder_filter_passing.csv', index=False)

if '_in_drive' in dir() and _in_drive:
    shutil.copy2('/content/forge_binder_filter_passing.csv',
                 os.path.join(NB02C_OUT, 'forge_binder_filter_passing.csv'))
    print(f'Saved to Drive: {NB02C_OUT}')

print(f'Saved: forge_binder_filter_passing.csv  ({len(passing)} rows)')
print(f'Saved: {len(os.listdir("/content/passing"))} CIF files to /content/passing/')
passing[out_cols].head(10)

## 10. Download

In [ ]:
import zipfile, shutil as _shutil

zip_name = '/content/forge_designs.zip'
with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write('/content/forge_binder_filter_passing.csv', 'forge_binder_filter_passing.csv')
    zf.write('/content/forge_pipeline_qc.png',           'forge_pipeline_qc.png')
    for fn in os.listdir('/content/passing'):
        zf.write(f'/content/passing/{fn}', f'structures/{fn}')

if '_in_drive' in dir() and _in_drive:
    _shutil.copy2(zip_name, os.path.join(NB02C_OUT, 'forge_designs.zip'))
    print(f'Saved to Drive: {NB02C_OUT}')

print(f'Zip ready: forge_designs.zip  ({os.path.getsize(zip_name)//1024} KB)')
print('Contents:')
print('  forge_binder_filter_passing.csv  <- upload to NB03')
print('  structures/*.cif')
print('\nNext: run NB03 with this CSV + all bindcraft_rankXX_passing.csv files')

try:
    from google.colab import files as _colab_files
    _colab_files.download(zip_name)
except ImportError:
    print(f'Not in Colab — file at: {zip_name}')